# 03 · Retrieval evaluation

Recall@1, Recall@5, MRR, nDCG@5 and latency for B0 (vector-only), E1 (original hybrid), E2 (improved hybrid) and E2S (E2 + evidence selection, local only); E3 (reranking) is not implemented. Headline results use the held-out set; the pilot set was used for tuning and is reported as a development set.

**Inputs** are versioned files under `research/data/` and `research/configs/`; nothing here connects to the production database or writes to it. Run top to bottom (Kernel → Restart & Run All).

In [ ]:
import sys, json, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

RESEARCH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "research"
sys.path.insert(0, str(RESEARCH / "scripts"))
import analysis as A

A.style()
np.random.seed(A.SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
EXPERIMENTS = json.loads((RESEARCH / "configs/experiments.json").read_text())
print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scipy {scipy.__version__} | matplotlib {matplotlib.__version__} | seed {A.SEED}")
print(f"dataset snapshot: {A.SNAPSHOT} (set RAG_SNAPSHOT=corpus_v1|corpus_v2) | results: {A.RESULTS.relative_to(RESEARCH)} | "
      f"figures: {A.FIGURES.relative_to(RESEARCH)}")

In [ ]:
frames = {}
for target in ("local", "production"):
    for ds in ("heldout_v0.2", "pilot_v0.1"):
        f = A.RESULTS / f"retrieval_{target}_{ds}.csv"
        if target == "production":   # erratum 2026-11-23: announcement ids corrected (scripts/rescore_production.py)
            f = A.RESULTS / f"retrieval_{target}_{ds}_rescored.csv"
        if f.exists():
            frames[(target, ds)] = A.load_retrieval(f.name)
avail = pd.DataFrame([(t, d, len(df), df["method"].nunique(), int((df["error"].fillna("") != "").sum()))
                      for (t, d), df in frames.items()], columns=["target", "dataset", "rows", "methods", "failed requests"])
avail

In [ ]:
display(pd.DataFrame([(k, v["name"], v.get("status", "run")) for k, v in EXPERIMENTS["methods"].items()],
                     columns=["id", "method", "status"]))
display(pd.DataFrame([(k, v["role"], v.get("review_status", v.get("status"))) for k, v in EXPERIMENTS["datasets"].items()],
                     columns=["dataset", "role", "review status"]))

## Results (95% cluster-bootstrap intervals over scenarios)

In [ ]:
order = ["B0", "E1", "E2", "E2S"]
tables = {}
for (t, d), df in frames.items():
    s = A.summarize(df, ["method"]); s.insert(0, "dataset", d); s.insert(0, "target", t)
    tables[(t, d)] = s.set_index("method").loc[[m for m in order if m in set(s["method"])]].reset_index()
results = pd.concat(tables.values(), ignore_index=True)
show = results[["target", "dataset", "method", "n", "r1", "r1_lo", "r1_hi", "r5", "rr", "rr_lo", "rr_hi", "ndcg5"]]
show.rename(columns={"r1": "R@1", "r5": "R@5", "rr": "MRR", "ndcg5": "nDCG@5"}).round(3)

In [ ]:
results.to_csv(A.RESULTS / "retrieval_summary.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, target in zip(axes, ["local", "production"]):
    key = (target, "heldout_v0.2")
    if key not in tables:
        ax.text(0.5, 0.5, "Not evaluated", ha="center", transform=ax.transAxes); ax.set_xticks([]); ax.set_yticks([]); continue
    t = tables[key].melt(id_vars=["method"], value_vars=["r1", "r5", "rr", "ndcg5"])
    rows = []
    for m in tables[key]["method"]:
        r = tables[key].set_index("method").loc[m]
        for metric, label in (("r1", "Recall@1"), ("r5", "Recall@5"), ("rr", "MRR"), ("ndcg5", "nDCG@5")):
            rows.append(dict(method=m, metric=label, v=r[metric], v_lo=r[f"{metric}_lo"], v_hi=r[f"{metric}_hi"]))
    tab = pd.DataFrame(rows).set_index(["method", "metric"])
    methods = list(tables[key]["method"])
    A.grouped_bars(ax, tab, ["Recall@1", "Recall@5", "MRR", "nDCG@5"], methods, "v")
    n = int(tables[key]["n"].iloc[0])
    ax.set_title(f"{target.capitalize()} · held-out v0.2 (n = {n} questions, 20 scenarios)")
axes[0].set_ylabel("score (95% CI)")
handles, labels = next((ax.get_legend_handles_labels() for ax in axes if ax.get_legend_handles_labels()[0]), ([], []))
fig.legend(handles, labels, loc="upper center", ncol=len(labels), bbox_to_anchor=(0.5, 1.07))
fig.tight_layout(); print(A.save(fig, "03_retrieval_baseline_comparison"))

## Paired comparisons

Same questions under both methods; the interval is a cluster bootstrap of the per-question difference; McNemar's exact test is shown for Recall@1. With 20 scenarios the test has low power - an interval that includes 0 means *no evidence of a difference*, not evidence of no difference.

In [ ]:
pairs = []
for (t, d), df in frames.items():
    for a, b in (("E2", "B0"), ("E2", "E1"), ("E1", "B0"), ("E2S", "E2")):
        if {a, b} <= set(df["method"]):
            for col in ("r1", "rr", "ndcg5"):
                r = A.paired_difference(df, a, b, col); r.update(target=t, dataset=d); pairs.append(r)
pairs = pd.DataFrame(pairs)
pairs["diff_pp"] = 100 * pairs["diff"]; pairs["ci_pp"] = pairs.apply(lambda r: f"[{100*r.lo:+.1f}, {100*r.hi:+.1f}]", axis=1)
pairs.to_csv(A.RESULTS / "retrieval_paired_comparisons.csv", index=False)
pairs[["target", "dataset", "a", "b", "metric", "n", "diff_pp", "ci_pp", "relative_pct", "a_better", "b_better",
       "mcnemar_p", "ci_excludes_zero"]].round(3)

## Latency

Local = in-memory index (no database, no network): measures ranking computation only. Production = HTTP round-trip to the deployed API with the real pgvector database (includes network).

In [ ]:
lat = pd.concat([df.assign(target=t, dataset=d) for (t, d), df in frames.items()])
lat = lat[lat["error"].fillna("") == ""]
lt = lat.groupby(["target", "method"])["latency_ms"].describe(percentiles=[0.5, 0.95])[["count", "50%", "95%", "mean"]]
display(lt.round(1))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, target in zip(axes, ["local", "production"]):
    sub = lat[lat["target"] == target]
    ms = [m for m in ["B0", "E1", "E2", "E2S"] if m in set(sub["method"])]
    if not ms:
        ax.text(0.5, 0.5, "Not evaluated", ha="center", transform=ax.transAxes); ax.set_xticks([]); ax.set_yticks([]); continue
    bp = ax.boxplot([sub.loc[sub["method"] == m, "latency_ms"] for m in ms], tick_labels=ms, patch_artist=True,
                    widths=0.5, showfliers=False, medianprops={"color": A.INK})
    for patch, m in zip(bp["boxes"], ms):
        patch.set_facecolor(A.COLORS[m]); patch.set_alpha(0.75)
    ax.set_title(f"{target.capitalize()} retrieval latency (n = {len(sub)} requests)"); ax.set_ylabel("milliseconds")
fig.tight_layout(); print(A.save(fig, "03_latency_distribution"))

In [ ]:
def fmt(t, d, a, b, col):
    r = pairs[(pairs.target == t) & (pairs.dataset == d) & (pairs.a == a) & (pairs.b == b) & (pairs.metric == col)]
    if r.empty: return "not evaluated"
    r = r.iloc[0]
    return f"{100*r['diff']:+.1f} pp (95% CI {100*r['lo']:+.1f} to {100*r['hi']:+.1f})"
text = []
for t in ("local", "production"):
    if (t, "heldout_v0.2") in frames:
        text.append(f"- **{t}, held-out**: E2 vs B0 Recall@1 {fmt(t, 'heldout_v0.2', 'E2', 'B0', 'r1')}, MRR "
                    f"{fmt(t, 'heldout_v0.2', 'E2', 'B0', 'rr')}; E2 vs E1 Recall@1 {fmt(t, 'heldout_v0.2', 'E2', 'E1', 'r1')}.")
    if (t, "pilot_v0.1") in frames:
        text.append(f"- **{t}, pilot (development set, used for tuning)**: E2 vs B0 Recall@1 "
                    f"{fmt(t, 'pilot_v0.1', 'E2', 'B0', 'r1')}.")
display(Markdown("**Interpretation.**\n\n" + "\n".join(text) + "\n\nThe gain on the pilot set is larger than on the "
                 "held-out set: the pilot set was used while tuning, so it overstates the improvement. The held-out "
                 "numbers are the ones to report, with their intervals; none of these labels is human-reviewed yet."))